# IFN580 Case Study 1 — Paddy Yield Classification

**Dataset:** `Asm1_dataset26.csv`

---

## Notebook Overview

This notebook walks through a complete binary-classification pipeline on a paddy-yield dataset.
The target variable `isAboveAvg` is **1** when a farm's yield per hectare exceeds the dataset mean, and **0** otherwise.

| Task | Focus |
|------|-------|
| **Task 1** | Data Preparation — cleaning, feature engineering, encoding, train/test split |
| **Task 2** | Decision Tree Modeling — default tree, hyperparameter tuning, feature importance |
| **Task 3** | Logistic Regression — full & reduced models, coefficient analysis, ROC |
| **Task 4** | Neural Networks — MLP, full & reduced models, loss curves, ROC |

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — safe for nbconvert
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

# Models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# Metrics
from sklearn.metrics import accuracy_score, roc_curve, auc, classification_report

# Paths (notebook lives in notebooks/, one level below project root)
ROOT      = Path("..").resolve()
RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
MODELS    = ROOT / "models"
MODELS.mkdir(exist_ok=True)
PROCESSED.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
print("All libraries imported.")
print(f"Project root : {ROOT}")

All libraries imported.
Project root : /Users/jishananam/paddy_project


## Task 1 — Data Preparation

Before any model can be trained the raw data must be cleaned and structured.
This task covers six steps:

1. **Load** the CSV and inspect its shape and types.
2. **Clean** — strip column-name whitespace; replace sentinel `'--'` values in the four Wind Direction columns with proper `NaN`s so they don't corrupt encoding later.
3. **Impute** — the four Min-temperature columns have sporadic missing values. We fill them with their column mean, a simple and interpretable strategy that preserves the distribution.
4. **Engineer the target** — raw Paddy yield (kg) is size-dependent. Dividing by Hectares gives `yield_per_hectare`, a fair cross-farm comparison. The binary label `isAboveAvg` is 1 when a farm exceeds the dataset mean.
5. **Encode** — tree-based and linear models both need numeric input. Ordinal-style columns (Variety, Soil Types, Nursery) get label encoding; the Wind Direction columns become one-hot dummies because their values are nominal with no natural ordering.
6. **Feature selection** — leaky columns (the raw yield/area figures and their ratio) are dropped. Pairs with Pearson |r| > 0.98 are pruned to one representative, eliminating multicollinearity before splitting.

In [2]:
# ── Task 1, Step 1: Load dataset ─────────────────────────────────────────────
df = pd.read_csv(RAW / "Asm1_dataset26.csv")
df.columns = df.columns.str.strip()          # strip leading/trailing whitespace
print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head())
df.info()

Loaded dataset: 2789 rows × 45 columns


,Hectares,Agriblock,Variety,Soil Types,Seedrate(in Kg),LP_Mainfield(in Tonnes),Nursery,Nursery area (Cents),LP_nurseryarea(in Tonnes),DAP_20days,...,Wind Direction_D1_D30,Wind Direction_D31_D60,Wind Direction_D61_D90,Wind Direction_D91_D120,Relative Humidity_D1_D30,Relative Humidity_D31_D60,Relative Humidity_D61_D90,Relative Humidity_D91_D120,Trash(in bundles),Paddy yield(in Kg)
0,6,Cuddalore,CO_43,alluvial,150,75.0,dry,120,6,240,...,SW,W,NNW,WSW,72.0,78,88,85,540,35028
1,6,Kurinjipadi,ponmani,clay,150,75.0,wet,120,6,240,...,NW,S,SE,SSE,64.6,85,84,87,600,35412
2,6,Panruti,delux ponni,alluvial,150,75.0,dry,120,6,240,...,ENE,NE,NNE,W,85.0,96,84,79,600,36300
3,6,Kallakurichi,CO_43,clay,150,75.0,wet,120,6,240,...,--,WNW,SE,S,88.5,95,81,84,540,35016
4,6,Sankarapuram,ponmani,alluvial,150,75.0,dry,120,6,240,...,SSE,W,SW,NW,72.7,91,83,81,600,34044


<class 'pandas.DataFrame'>
RangeIndex: 2789 entries, 0 to 2788
Data columns (total 45 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Hectares                            2789 non-null   int64  
 1   Agriblock                           2789 non-null   str    
 2   Variety                             2789 non-null   str    
 3   Soil Types                          2789 non-null   str    
 4   Seedrate(in Kg)                     2789 non-null   int64  
 5   LP_Mainfield(in Tonnes)             2789 non-null   float64
 6   Nursery                             2789 non-null   str    
 7   Nursery area (Cents)                2789 non-null   int64  
 8   LP_nurseryarea(in Tonnes)           2789 non-null   int64  
 9   DAP_20days                          2789 non-null   int64  
 10  Weed28D_thiobencarb                 2789 non-null   int64  
 11  Urea_40Days                         2789 non-null   fl

In [3]:
# ── Task 1, Step 2: Replace '--' in Wind Direction columns with NaN ──────────
wind_cols = [
    "Wind Direction_D1_D30",
    "Wind Direction_D31_D60",
    "Wind Direction_D61_D90",
    "Wind Direction_D91_D120",
]
for col in wind_cols:
    before = (df[col] == "--").sum()
    df[col] = df[col].replace("--", np.nan)
    print(f"  Replaced {before} '--' values in '{col}' with NaN")

# ── Task 1, Step 3: Impute missing Min temp columns with column mean ─────────
min_temp_cols = [
    "Min temp_D1_D30",
    "Min temp_D31_D60",
    "Min temp_D61_D90",
    "Min temp_D91_D120",
]
print("\nImputing missing Min temp values with column mean:")
for col in min_temp_cols:
    missing = df[col].isna().sum()
    col_mean = df[col].mean()
    df[col] = df[col].fillna(col_mean)
    print(f"  '{col}': filled {missing} NaN(s) with mean={col_mean:.4f}")

  Replaced 691 '--' values in 'Wind Direction_D1_D30' with NaN
  Replaced 691 '--' values in 'Wind Direction_D31_D60' with NaN
  Replaced 691 '--' values in 'Wind Direction_D61_D90' with NaN
  Replaced 691 '--' values in 'Wind Direction_D91_D120' with NaN

Imputing missing Min temp values with column mean:
  'Min temp_D1_D30': filled 108 NaN(s) with mean=19.3316
  'Min temp_D31_D60': filled 108 NaN(s) with mean=17.1371
  'Min temp_D61_D90': filled 108 NaN(s) with mean=16.6766
  'Min temp_D91_D120': filled 108 NaN(s) with mean=16.1833


In [4]:
# ── Task 1, Step 4: Calculate yield_per_hectare and create binary target ─────
df["yield_per_hectare"] = df["Paddy yield(in Kg)"] / df["Hectares"]
print(f"yield_per_hectare — min: {df['yield_per_hectare'].min():.2f}, "
      f"max: {df['yield_per_hectare'].max():.2f}, "
      f"mean: {df['yield_per_hectare'].mean():.2f}")

mean_yield = df["yield_per_hectare"].mean()
df["isAboveAvg"] = (df["yield_per_hectare"] > mean_yield).astype(int)

print("\nisAboveAvg value counts:")
print(df["isAboveAvg"].value_counts().to_string())
total = len(df)
for label, count in df["isAboveAvg"].value_counts().sort_index().items():
    tag = "Above avg" if label == 1 else "Below avg"
    print(f"  {tag} ({label}): {count:>5}  ({100*count/total:.1f}%)")
print(f"  (threshold: yield_per_hectare > {mean_yield:.2f} kg/ha)")

yield_per_hectare — min: 5410.00, max: 6469.00, mean: 5990.53

isAboveAvg value counts:
isAboveAvg
0    1415
1    1374
  Below avg (0):  1415  (50.7%)
  Above avg (1):  1374  (49.3%)
  (threshold: yield_per_hectare > 5990.53 kg/ha)


In [5]:
# ── Task 1, Step 5: Encode categorical columns ───────────────────────────────
label_cols = ["Variety", "Soil Types", "Nursery"]
le = LabelEncoder()
print("Label encoding:")
for col in label_cols:
    original = df[col].unique().tolist()
    df[col] = le.fit_transform(df[col])
    encoded = sorted(df[col].unique().tolist())
    print(f"  '{col}': {original} → {encoded}")

# One-hot encode Wind Direction columns (nominal, no ordinal relationship)
print("\nOne-hot encoding Wind Direction columns:")
before_cols = df.shape[1]
df = pd.get_dummies(df, columns=wind_cols, dummy_na=False)
ohe_cols = [c for c in df.columns if c.startswith("Wind Direction_")]
print(f"  Created {len(ohe_cols)} one-hot columns: {ohe_cols}")
print(f"  Shape after OHE: {df.shape}")

Label encoding:
  'Variety': ['CO_43', 'ponmani', 'delux ponni'] → [0, 1, 2]
  'Soil Types': ['alluvial', 'clay'] → [0, 1]
  'Nursery': ['dry', 'wet'] → [0, 1]

One-hot encoding Wind Direction columns:
  Created 22 one-hot columns: ['Wind Direction_D1_D30_E', 'Wind Direction_D1_D30_ENE', 'Wind Direction_D1_D30_NW', 'Wind Direction_D1_D30_SSE', 'Wind Direction_D1_D30_SW', 'Wind Direction_D1_D30_W', 'Wind Direction_D31_D60_ENE', 'Wind Direction_D31_D60_NE', 'Wind Direction_D31_D60_S', 'Wind Direction_D31_D60_W', 'Wind Direction_D31_D60_WNW', 'Wind Direction_D61_D90_NE', 'Wind Direction_D61_D90_NNE', 'Wind Direction_D61_D90_NNW', 'Wind Direction_D61_D90_SE', 'Wind Direction_D61_D90_SW', 'Wind Direction_D91_D120_NNW', 'Wind Direction_D91_D120_NW', 'Wind Direction_D91_D120_S', 'Wind Direction_D91_D120_SSE', 'Wind Direction_D91_D120_W', 'Wind Direction_D91_D120_WSW']
  Shape after OHE: (2789, 65)


In [6]:
# ── Task 1, Step 6a: Drop leaky and irrelevant columns ───────────────────────
# "Paddy yield(in Kg)" and "Hectares" directly determine the target — including
# them would be data leakage. "yield_per_hectare" was only needed to create the
# target. "Agriblock" is an administrative ID with no predictive value.
drop_cols = ["Hectares", "Paddy yield(in Kg)", "yield_per_hectare", "Agriblock"]
df.drop(columns=drop_cols, inplace=True)
print(f"Dropped columns: {drop_cols}")
print(f"Shape after drop: {df.shape}")

# ── Task 1, Step 6b: Drop highly correlated columns (r > 0.98) ───────────────
# High pairwise correlation (near-duplicate features) inflates model complexity
# without adding information and can destabilise logistic regression coefficients.
X_temp = df.drop(columns=["isAboveAvg"])
corr_matrix = X_temp.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.98)]
if to_drop:
    print(f"\nDropping {len(to_drop)} highly correlated column(s) (r > 0.98):")
    for col in to_drop:
        partners = upper.index[upper[col] > 0.98].tolist()
        print(f"  '{col}' correlated with: {partners}")
    df.drop(columns=to_drop, inplace=True)
else:
    print("\nNo columns found with pairwise correlation > 0.98")
print(f"Shape after correlation drop: {df.shape}")

# ── Correlation heatmap ───────────────────────────────────────────────────────
numeric_df = df.drop(columns=["isAboveAvg"]).select_dtypes(include=[np.number])
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(numeric_df.corr(), annot=False, cmap="coolwarm", center=0,
            linewidths=0.2, ax=ax)
ax.set_title("Feature Correlation Heatmap (post-pruning)", fontsize=13)
plt.tight_layout()
plt.savefig(MODELS / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved models/correlation_heatmap.png")

Dropped columns: ['Hectares', 'Paddy yield(in Kg)', 'yield_per_hectare', 'Agriblock']
Shape after drop: (2789, 61)

Dropping 15 highly correlated column(s) (r > 0.98):
  'LP_Mainfield(in Tonnes)' correlated with: ['Seedrate(in Kg)']
  'Nursery area (Cents)' correlated with: ['Seedrate(in Kg)', 'LP_Mainfield(in Tonnes)']
  'LP_nurseryarea(in Tonnes)' correlated with: ['Seedrate(in Kg)', 'LP_Mainfield(in Tonnes)', 'Nursery area (Cents)']
  'DAP_20days' correlated with: ['Seedrate(in Kg)', 'LP_Mainfield(in Tonnes)', 'Nursery area (Cents)', 'LP_nurseryarea(in Tonnes)']
  'Weed28D_thiobencarb' correlated with: ['Seedrate(in Kg)', 'LP_Mainfield(in Tonnes)', 'Nursery area (Cents)', 'LP_nurseryarea(in Tonnes)', 'DAP_20days']
  'Urea_40Days' correlated with: ['Seedrate(in Kg)', 'LP_Mainfield(in Tonnes)', 'Nursery area (Cents)', 'LP_nurseryarea(in Tonnes)', 'DAP_20days', 'Weed28D_thiobencarb']
  'Potassh_50Days' correlated with: ['Seedrate(in Kg)', 'LP_Mainfield(in Tonnes)', 'Nursery area (Cents

Saved models/correlation_heatmap.png


/var/folders/y2/j5bvysgd7477ltmvzm3wf4xr0000gn/T/ipykernel_19403/2939127766.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Task 1, Step 7: Train/test split and save ────────────────────────────────
X = df.drop(columns=["isAboveAvg"])
y = df["isAboveAvg"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Final split shapes:")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"  y_test  : {y_test.shape}")

# Persist splits so subsequent tasks can load them independently
X_train.to_csv(PROCESSED / "X_train.csv", index=False)
X_test.to_csv( PROCESSED / "X_test.csv",  index=False)
y_train.to_csv(PROCESSED / "y_train.csv", index=False)
y_test.to_csv( PROCESSED / "y_test.csv",  index=False)
print(f"\nSaved splits to {PROCESSED}/")
print("  X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("\nTask 1 complete.")

Final split shapes:
  X_train : (2231, 45)
  X_test  : (558, 45)
  y_train : (2231,)
  y_test  : (558,)

Saved splits to /Users/jishananam/paddy_project/data/processed/
  X_train.csv, X_test.csv, y_train.csv, y_test.csv

Task 1 complete.


## Task 2 — Decision Tree Modeling

Decision trees are interpretable, non-parametric classifiers that recursively partition the feature space.
We run two experiments here:

- **Default tree** — trained with sklearn defaults (no depth limit). This gives a baseline and reveals which features the tree considers most informative at each split.
- **Tuned tree** — we search a grid of `max_depth`, `min_samples_split`, `min_samples_leaf`, and `criterion` using 5-fold cross-validation. Restricting depth and increasing leaf-size requirements acts as regularisation, trading some training-accuracy for better generalisation.

The top-10 features from the **best tuned tree** are then carried forward as the reduced feature set in Tasks 3 and 4.

In [8]:
# ── Task 2, Step 1: Load processed splits ────────────────────────────────────
X_train = pd.read_csv(PROCESSED / "X_train.csv")
X_test  = pd.read_csv(PROCESSED / "X_test.csv")
y_train = pd.read_csv(PROCESSED / "y_train.csv").squeeze()
y_test  = pd.read_csv(PROCESSED / "y_test.csv").squeeze()
feature_names = X_train.columns.tolist()
print(f"Loaded — X_train: {X_train.shape}, X_test: {X_test.shape}")

Loaded — X_train: (2231, 45), X_test: (558, 45)


In [9]:
# ── Task 2, Step 2: Default Decision Tree ────────────────────────────────────
print("=" * 60)
print("DEFAULT DECISION TREE")
print("=" * 60)
dt_default = DecisionTreeClassifier(random_state=42)
dt_default.fit(X_train, y_train)

train_acc = accuracy_score(y_train, dt_default.predict(X_train))
test_acc  = accuracy_score(y_test,  dt_default.predict(X_test))
print(f"Training accuracy : {train_acc:.4f}")
print(f"Test accuracy     : {test_acc:.4f}")
print(f"Number of nodes   : {dt_default.tree_.node_count}")
print(f"Max depth         : {dt_default.get_depth()}")

# ── First and second split variables ─────────────────────────────────────────
tree_ = dt_default.tree_
root_feature  = feature_names[tree_.feature[0]]
left_child    = tree_.children_left[0]
right_child   = tree_.children_right[0]
left_feat     = feature_names[tree_.feature[left_child]]  if tree_.feature[left_child]  >= 0 else "leaf"
right_feat    = feature_names[tree_.feature[right_child]] if tree_.feature[right_child] >= 0 else "leaf"

print(f"\n1st split variable: '{root_feature}'")
print(f"2nd split variable: left branch  → '{left_feat}'")
print(f"                    right branch → '{right_feat}'")

# ── Top 10 feature importances ────────────────────────────────────────────────
importances   = pd.Series(dt_default.feature_importances_, index=feature_names)
top10_default = importances.nlargest(10)
print("\nTop 10 feature importances (default):")
for feat, imp in top10_default.items():
    print(f"  {feat:<45} {imp:.6f}")

DEFAULT DECISION TREE
Training accuracy : 0.9390
Test accuracy     : 0.8441
Number of nodes   : 545
Max depth         : 16

1st split variable: 'Trash(in bundles)'
2nd split variable: left branch  → 'Wind Direction_D31_D60_W'
                    right branch → 'Trash(in bundles)'

Top 10 feature importances (default):
  Trash(in bundles)                             0.854201
  Seedrate(in Kg)                               0.022565
  Soil Types                                    0.022016
  Nursery                                       0.012448
  Variety                                       0.012072
  Relative Humidity_D31_D60                     0.004778
  Inst Wind Speed_D1_D30(in Knots)              0.004499
  Wind Direction_D61_D90_SE                     0.004059
  Wind Direction_D61_D90_NNW                    0.004030
  Wind Direction_D1_D30_SSE                     0.003420


In [10]:
# ── Task 2, Step 3: Tuned Decision Tree with GridSearchCV ────────────────────
print("=" * 60)
print("TUNED DECISION TREE (GridSearchCV)")
print("=" * 60)
param_grid = {
    "max_depth"        : [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf" : [1, 2, 5, 10],
    "criterion"        : ["gini", "entropy"],
}
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X_train, y_train)
dt_tuned = grid_search.best_estimator_

print("Best parameters:")
for k, v in grid_search.best_params_.items():
    print(f"  {k:<22}: {v}")
print(f"Best CV accuracy  : {grid_search.best_score_:.4f}")

train_acc_t = accuracy_score(y_train, dt_tuned.predict(X_train))
test_acc_t  = accuracy_score(y_test,  dt_tuned.predict(X_test))
print(f"Training accuracy : {train_acc_t:.4f}")
print(f"Test accuracy     : {test_acc_t:.4f}")
print(f"Number of nodes   : {dt_tuned.tree_.node_count}")
print(f"Max depth         : {dt_tuned.get_depth()}")

# ── Top 10 feature importances (tuned) ───────────────────────────────────────
importances_t  = pd.Series(dt_tuned.feature_importances_, index=feature_names)
top10_tuned    = importances_t.nlargest(10)
top10_dt_features = top10_tuned.index.tolist()   # <-- used by Tasks 3 & 4

print("\nTop 10 feature importances (tuned):")
for feat, imp in top10_tuned.items():
    print(f"  {feat:<45} {imp:.6f}")

print(f"\ntop10_dt_features = {top10_dt_features}")

TUNED DECISION TREE (GridSearchCV)


Best parameters:
  criterion             : entropy
  max_depth             : 5
  min_samples_leaf      : 1
  min_samples_split     : 2
Best CV accuracy  : 0.9117
Training accuracy : 0.9148
Test accuracy     : 0.8961
Number of nodes   : 39
Max depth         : 5

Top 10 feature importances (tuned):
  Trash(in bundles)                             0.921036
  Seedrate(in Kg)                               0.044149
  Variety                                       0.013118
  Soil Types                                    0.004952
  Relative Humidity_D1_D30                      0.004595
  Max temp_D31_D60                              0.003583
  Inst Wind Speed_D61_D90(in Knots)             0.003164
  Wind Direction_D1_D30_W                       0.001707
  Inst Wind Speed_D1_D30(in Knots)              0.001234
  Min temp_D31_D60                              0.000998

top10_dt_features = ['Trash(in bundles)', 'Seedrate(in Kg)', 'Variety', 'Soil Types', 'Relative Humidity_D1_D30', 'Max temp_D31_D60

In [11]:
# ── Task 2, Step 4: Visualise both trees and ROC curves ──────────────────────
print("Saving tree visualisations...")
for label, model, fname in [
    ("Default", dt_default, "tree_default.png"),
    ("Tuned",   dt_tuned,   "tree_tuned.png"),
]:
    fig, ax = plt.subplots(figsize=(24, 8))
    plot_tree(
        model,
        max_depth=3,
        feature_names=feature_names,
        class_names=["BelowAvg", "AboveAvg"],
        filled=True,
        rounded=True,
        fontsize=8,
        ax=ax,
    )
    ax.set_title(f"{label} Decision Tree (max_depth=3 view)", fontsize=14)
    fig.tight_layout()
    fig.savefig(MODELS / fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved models/{fname}")

# ── ROC curves ────────────────────────────────────────────────────────────────
print("\nSaving ROC curve plot...")
fig, ax = plt.subplots(figsize=(8, 6))
for label, model, color in [
    ("Default DT", dt_default, "steelblue"),
    ("Tuned DT",   dt_tuned,   "darkorange"),
]:
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{label} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier")
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Default vs Tuned Decision Tree", fontsize=13)
ax.legend(loc="lower right", fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(MODELS / "roc_decision_trees.png", dpi=150, bbox_inches="tight")
plt.show()
print("  Saved models/roc_decision_trees.png")
print("\nTask 2 complete.")

Saving tree visualisations...


/var/folders/y2/j5bvysgd7477ltmvzm3wf4xr0000gn/T/ipykernel_19403/3925365998.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  Saved models/tree_default.png
  Saved models/tree_tuned.png

Saving ROC curve plot...


  Saved models/roc_decision_trees.png

Task 2 complete.


/var/folders/y2/j5bvysgd7477ltmvzm3wf4xr0000gn/T/ipykernel_19403/3925365998.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Task 3 — Logistic Regression Modeling

Logistic Regression fits a linear decision boundary in feature space and outputs calibrated probabilities.
It is a strong baseline before moving to neural networks because its coefficients are directly interpretable as log-odds contributions per feature.

Two models are compared:

- **Full model** — uses every feature from the processed dataset. `StandardScaler` is applied because gradient-based solvers (lbfgs) converge much faster when features are on comparable scales.  
- **Reduced model** — uses only the top-10 features identified by the best Decision Tree from Task 2. A smaller feature set reduces variance and is easier to interpret, at the potential cost of a small accuracy drop.

Both models are tuned with 5-fold GridSearchCV over `C` (regularisation strength), `solver`, and `max_iter`.
ROC curves for both are overlaid to compare discrimination ability at every threshold.

In [12]:
# ── Task 3, Step 1: Load splits and apply StandardScaler ─────────────────────
X_train = pd.read_csv(PROCESSED / "X_train.csv")
X_test  = pd.read_csv(PROCESSED / "X_test.csv")
y_train = pd.read_csv(PROCESSED / "y_train.csv").squeeze()
y_test  = pd.read_csv(PROCESSED / "y_test.csv").squeeze()
print(f"Loaded — X_train: {X_train.shape}, X_test: {X_test.shape}")

# Fit scaler on training data only to avoid data leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("StandardScaler applied (fit on train, transform on both).")

Loaded — X_train: (2231, 45), X_test: (558, 45)
StandardScaler applied (fit on train, transform on both).


In [13]:
# ── Task 3, Step 2: Full Logistic Regression with GridSearchCV ───────────────
print("=" * 60)
print("FULL LOGISTIC REGRESSION (GridSearchCV)")
print("=" * 60)
lr_param_grid = {
    "C"       : [0.01, 0.1, 1, 10, 100],
    "solver"  : ["lbfgs", "liblinear"],
    "max_iter": [200, 500, 1000],
}
lr_grid = GridSearchCV(
    LogisticRegression(random_state=42),
    lr_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)
lr_grid.fit(X_train_sc, y_train)
lr_full = lr_grid.best_estimator_

print("Best parameters:")
for k, v in lr_grid.best_params_.items():
    print(f"  {k:<10}: {v}")
print(f"Best CV accuracy  : {lr_grid.best_score_:.4f}")

train_acc_lr = accuracy_score(y_train, lr_full.predict(X_train_sc))
test_acc_lr  = accuracy_score(y_test,  lr_full.predict(X_test_sc))
print(f"Training accuracy : {train_acc_lr:.4f}")
print(f"Test accuracy     : {test_acc_lr:.4f}")

# ── Top 10 features by absolute coefficient value ────────────────────────────
coef_series = pd.Series(np.abs(lr_full.coef_[0]), index=X_train.columns)
top10_lr    = coef_series.nlargest(10)
print("\nTop 10 features by |coefficient|:")
for feat, coef in top10_lr.items():
    print(f"  {feat:<45} {coef:.6f}")

FULL LOGISTIC REGRESSION (GridSearchCV)


Best parameters:
  C         : 0.01
  max_iter  : 200
  solver    : liblinear
Best CV accuracy  : 0.8862
Training accuracy : 0.8902
Test accuracy     : 0.8710

Top 10 features by |coefficient|:
  Seedrate(in Kg)                               1.094845
  Trash(in bundles)                             0.692883
  Variety                                       0.159184
  Nursery                                       0.049698
  Wind Direction_D91_D120_W                     0.047962
  Wind Direction_D31_D60_NE                     0.040500
  Wind Direction_D1_D30_W                       0.036904
  Wind Direction_D1_D30_ENE                     0.025888
  Wind Direction_D31_D60_WNW                    0.021751
  Min temp_D1_D30                               0.020814


In [14]:
# ── Task 3, Step 3: Reduced Logistic Regression (top-10 DT features) ─────────
# top10_dt_features was set in Task 2; using the decision tree's ranking
# as a principled feature-selection method for a different model family.
print("=" * 60)
print("REDUCED LOGISTIC REGRESSION (top-10 DT features)")
print("=" * 60)
print(f"Features used: {top10_dt_features}")

X_train_red = X_train[top10_dt_features]
X_test_red  = X_test[top10_dt_features]

scaler_red   = StandardScaler()
X_train_red_sc = scaler_red.fit_transform(X_train_red)
X_test_red_sc  = scaler_red.transform(X_test_red)

lr_grid_red = GridSearchCV(
    LogisticRegression(random_state=42),
    lr_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)
lr_grid_red.fit(X_train_red_sc, y_train)
lr_reduced = lr_grid_red.best_estimator_

print("\nBest parameters:")
for k, v in lr_grid_red.best_params_.items():
    print(f"  {k:<10}: {v}")
print(f"Best CV accuracy  : {lr_grid_red.best_score_:.4f}")

train_acc_lrr = accuracy_score(y_train, lr_reduced.predict(X_train_red_sc))
test_acc_lrr  = accuracy_score(y_test,  lr_reduced.predict(X_test_red_sc))
print(f"Training accuracy : {train_acc_lrr:.4f}")
print(f"Test accuracy     : {test_acc_lrr:.4f}")

REDUCED LOGISTIC REGRESSION (top-10 DT features)
Features used: ['Trash(in bundles)', 'Seedrate(in Kg)', 'Variety', 'Soil Types', 'Relative Humidity_D1_D30', 'Max temp_D31_D60', 'Inst Wind Speed_D61_D90(in Knots)', 'Wind Direction_D1_D30_W', 'Inst Wind Speed_D1_D30(in Knots)', 'Min temp_D31_D60']

Best parameters:
  C         : 0.01
  max_iter  : 200
  solver    : liblinear
Best CV accuracy  : 0.8902
Training accuracy : 0.8911
Test accuracy     : 0.8746


In [15]:
# ── Task 3, Step 4: ROC curves — Full vs Reduced Logistic Regression ─────────
fig, ax = plt.subplots(figsize=(8, 6))
for label, model, X_sc, color in [
    ("LR Full",    lr_full,    X_test_sc,     "steelblue"),
    ("LR Reduced", lr_reduced, X_test_red_sc, "darkorange"),
]:
    y_prob = model.predict_proba(X_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{label} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier")
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Logistic Regression (Full vs Reduced)", fontsize=13)
ax.legend(loc="lower right", fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(MODELS / "roc_logistic_regression.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved models/roc_logistic_regression.png")
print("\nTask 3 complete.")

Saved models/roc_logistic_regression.png

Task 3 complete.


/var/folders/y2/j5bvysgd7477ltmvzm3wf4xr0000gn/T/ipykernel_19403/1099301255.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Task 4 — Neural Network Modeling

Multi-layer Perceptrons (MLPs) can learn non-linear decision boundaries that linear models miss.
Feature scaling is **essential** for neural networks: without it, units in early layers receive gradients of wildly different magnitudes, causing slow or unstable training.

Two models are compared:

- **Full model** — trains on all processed features with several hidden-layer configurations. The loss curve lets us verify that the network actually converged before the `max_iter` budget ran out.
- **Reduced model** — same grid search but restricted to the top-10 features from the best Decision Tree (Task 2). Fewer inputs can reduce overfitting and training time for small datasets.

`GridSearchCV` with 5-fold CV explores hidden-layer size, activation function, L2 regularisation (`alpha`), and maximum iterations. ROC curves for both networks are plotted together for direct AUC comparison.

In [16]:
# ── Task 4, Step 1: Load splits and apply StandardScaler ─────────────────────
X_train = pd.read_csv(PROCESSED / "X_train.csv")
X_test  = pd.read_csv(PROCESSED / "X_test.csv")
y_train = pd.read_csv(PROCESSED / "y_train.csv").squeeze()
y_test  = pd.read_csv(PROCESSED / "y_test.csv").squeeze()
print(f"Loaded — X_train: {X_train.shape}, X_test: {X_test.shape}")

# Neural networks are sensitive to feature scale; always fit scaler on train only
nn_scaler   = StandardScaler()
X_train_sc  = nn_scaler.fit_transform(X_train)
X_test_sc   = nn_scaler.transform(X_test)
print("StandardScaler applied.")

Loaded — X_train: (2231, 45), X_test: (558, 45)
StandardScaler applied.


In [17]:
# ── Task 4, Step 2: Full Neural Network with GridSearchCV ────────────────────
print("=" * 60)
print("FULL NEURAL NETWORK (GridSearchCV)")
print("=" * 60)
nn_param_grid = {
    "hidden_layer_sizes": [(50,), (100,), (50, 50), (100, 50)],
    "activation"        : ["relu", "tanh"],
    "alpha"             : [0.0001, 0.001, 0.01],
    "max_iter"          : [200, 500],
}
nn_grid = GridSearchCV(
    MLPClassifier(random_state=42, early_stopping=False),
    nn_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)
nn_grid.fit(X_train_sc, y_train)
nn_full = nn_grid.best_estimator_

print("Best parameters:")
for k, v in nn_grid.best_params_.items():
    print(f"  {k:<22}: {v}")
print(f"Best CV accuracy  : {nn_grid.best_score_:.4f}")

train_acc_nn = accuracy_score(y_train, nn_full.predict(X_train_sc))
test_acc_nn  = accuracy_score(y_test,  nn_full.predict(X_test_sc))
print(f"Training accuracy : {train_acc_nn:.4f}")
print(f"Test accuracy     : {test_acc_nn:.4f}")

# ── Check convergence ────────────────────────────────────────────────────────
converged = nn_full.n_iter_ < nn_full.max_iter
print(f"\nIterations run    : {nn_full.n_iter_} / {nn_full.max_iter}")
print(f"Converged         : {converged}")

FULL NEURAL NETWORK (GridSearchCV)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


...

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Best parameters:
  activation            : tanh
  alpha                 : 0.01
  hidden_layer_sizes    : (100,)
  max_iter              : 200
Best CV accuracy  : 0.8942
Training accuracy : 0.9247
Test accuracy     : 0.8853

Iterations run    : 200 / 200
Converged         : False


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [18]:
# ── Task 4, Step 3: Loss curve — Full Neural Network ─────────────────────────
# loss_curve_ is only populated when the best estimator is re-fitted on the
# full training set (which GridSearchCV does automatically for best_estimator_).
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(nn_full.loss_curve_, color="steelblue", lw=2, label="Training loss")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss (cross-entropy)", fontsize=12)
ax.set_title("Full Neural Network — Training Loss Curve", fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(MODELS / "nn_loss_curve_full.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved models/nn_loss_curve_full.png")

Saved models/nn_loss_curve_full.png


/var/folders/y2/j5bvysgd7477ltmvzm3wf4xr0000gn/T/ipykernel_19403/1726341977.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# ── Task 4, Step 4: Reduced Neural Network (top-10 DT features) ──────────────
print("=" * 60)
print("REDUCED NEURAL NETWORK (top-10 DT features)")
print("=" * 60)
print(f"Features used: {top10_dt_features}")

X_train_red    = X_train[top10_dt_features]
X_test_red     = X_test[top10_dt_features]
nn_scaler_red  = StandardScaler()
X_train_red_sc = nn_scaler_red.fit_transform(X_train_red)
X_test_red_sc  = nn_scaler_red.transform(X_test_red)

nn_grid_red = GridSearchCV(
    MLPClassifier(random_state=42, early_stopping=False),
    nn_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)
nn_grid_red.fit(X_train_red_sc, y_train)
nn_reduced = nn_grid_red.best_estimator_

print("\nBest parameters:")
for k, v in nn_grid_red.best_params_.items():
    print(f"  {k:<22}: {v}")
print(f"Best CV accuracy  : {nn_grid_red.best_score_:.4f}")

train_acc_nnr = accuracy_score(y_train, nn_reduced.predict(X_train_red_sc))
test_acc_nnr  = accuracy_score(y_test,  nn_reduced.predict(X_test_red_sc))
print(f"Training accuracy : {train_acc_nnr:.4f}")
print(f"Test accuracy     : {test_acc_nnr:.4f}")
print(f"Iterations run    : {nn_reduced.n_iter_} / {nn_reduced.max_iter}")
print(f"Converged         : {nn_reduced.n_iter_ < nn_reduced.max_iter}")

REDUCED NEURAL NETWORK (top-10 DT features)
Features used: ['Trash(in bundles)', 'Seedrate(in Kg)', 'Variety', 'Soil Types', 'Relative Humidity_D1_D30', 'Max temp_D31_D60', 'Inst Wind Speed_D61_D90(in Knots)', 'Wind Direction_D1_D30_W', 'Inst Wind Speed_D1_D30(in Knots)', 'Min temp_D31_D60']


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


...

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iteration


Best parameters:
  activation            : tanh
  alpha                 : 0.0001
  hidden_layer_sizes    : (50, 50)
  max_iter              : 200
Best CV accuracy  : 0.9117
Training accuracy : 0.9144
Test accuracy     : 0.8907
Iterations run    : 111 / 200
Converged         : True


In [20]:
# ── Task 4, Step 5: Loss curve — Reduced Neural Network ──────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(nn_reduced.loss_curve_, color="darkorange", lw=2, label="Training loss (reduced)")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss (cross-entropy)", fontsize=12)
ax.set_title("Reduced Neural Network — Training Loss Curve", fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(MODELS / "nn_loss_curve_reduced.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved models/nn_loss_curve_reduced.png")

Saved models/nn_loss_curve_reduced.png


/var/folders/y2/j5bvysgd7477ltmvzm3wf4xr0000gn/T/ipykernel_19403/692670296.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# ── Task 4, Step 6: ROC curves — Full vs Reduced Neural Network ──────────────
fig, ax = plt.subplots(figsize=(8, 6))
for label, model, X_sc, color in [
    ("NN Full",    nn_full,    X_test_sc,     "steelblue"),
    ("NN Reduced", nn_reduced, X_test_red_sc, "darkorange"),
]:
    y_prob = model.predict_proba(X_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{label} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier")
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Neural Networks (Full vs Reduced)", fontsize=13)
ax.legend(loc="lower right", fontsize=11)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(MODELS / "roc_neural_networks.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved models/roc_neural_networks.png")
print("\nTask 4 complete.")

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"{'Model':<30} {'Train Acc':>10} {'Test Acc':>10}")
print("-" * 52)
print(f"{'DT Default':<30} {train_acc:>10.4f} {test_acc:>10.4f}")
print(f"{'DT Tuned':<30} {train_acc_t:>10.4f} {test_acc_t:>10.4f}")
print(f"{'LR Full':<30} {train_acc_lr:>10.4f} {test_acc_lr:>10.4f}")
print(f"{'LR Reduced (top-10 DT)':<30} {train_acc_lrr:>10.4f} {test_acc_lrr:>10.4f}")
print(f"{'NN Full':<30} {train_acc_nn:>10.4f} {test_acc_nn:>10.4f}")
print(f"{'NN Reduced (top-10 DT)':<30} {train_acc_nnr:>10.4f} {test_acc_nnr:>10.4f}")

Saved models/roc_neural_networks.png

Task 4 complete.

RESULTS SUMMARY
Model                           Train Acc   Test Acc
----------------------------------------------------
DT Default                         0.9390     0.8441
DT Tuned                           0.9148     0.8961
LR Full                            0.8902     0.8710
LR Reduced (top-10 DT)             0.8911     0.8746
NN Full                            0.9247     0.8853
NN Reduced (top-10 DT)             0.9144     0.8907


/var/folders/y2/j5bvysgd7477ltmvzm3wf4xr0000gn/T/ipykernel_19403/140339736.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
